[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navjotts/ML-experiments/blob/master/35%20-%20Common%20mistake%20in%20creating%20Word%20Embeddings/Common_mistake_in_creating_Word_Embeddings.ipynb)

# Overview

## word2vec
<img alt="word2vec" src="https://www.dropbox.com/s/wdimfrk55t44ahy/word2vec.png?raw=1" width="70%" height="70%">

### 2 main architectures

*   The CBOW architecture predicts the current word based on the context
*   The Skip-gram architecture predicts the surroundng words given the current word

---

**Let's see a simple mistake which can be made if we don't understand properly what is happening behind the scenes**


# Data

We'll use the 20 newsgroups text dataset from sklearn

In [1]:
from sklearn.datasets import fetch_20newsgroups
dataset = fetch_20newsgroups(subset='train', 
                             remove=('headers', 'footers'), 
                             shuffle=True)

In [15]:
print("%d articles" % len(dataset.data))

11314 articles


In [0]:
import nltk
nltk.download('all')

In [0]:
!pip install --upgrade gensim

# Tokenization 

Helper functions to tokenize as per need

In [0]:
from enum import Enum
class Wordbase(Enum):
  LEMMA = 0
  STEM = 1
  TEXT = 2

import string
def remove_punctuation(str):
  return str.translate(str.maketrans("", "", string.punctuation))

from nltk import pos_tag
import nltk.corpus.reader.wordnet as wordnet
def pos_for_WordNetLemmatizer(token):  
  word, tag = pos_tag([token])[0]  
  if tag in ['VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ']:
    return wordnet.VERB
  elif tag in ['JJ', 'JJR', 'JJS']:
    return wordnet.ADJ
  elif tag in ['RB', 'RBR', 'RBS']:
    return wordnet.ADV  
  return wordnet.NOUN
  
from nltk import word_tokenize
from nltk import WordNetLemmatizer
from nltk import PorterStemmer
def process_and_tokenize(str, word_base, remove_stop=True):
  stopwords = nltk.corpus.stopwords.words('english')
  lemmatizer = WordNetLemmatizer()
  stemmer = PorterStemmer()  
  tokens = []
  for t in word_tokenize(str):
    updated_t = remove_punctuation(t)
    if len(updated_t) > 0 and (not remove_stop or t.lower() not in stopwords):
      if (word_base == Wordbase.LEMMA):
        tokens.append(lemmatizer.lemmatize(updated_t, pos_for_WordNetLemmatizer(updated_t)))
      elif (word_base == Wordbase.STEM):
        tokens.append(stemmer.stem(updated_t))
      else:
        tokens.append(updated_t)  
  return tokens

**Tokenize the body of text for each article**

In [7]:
tokenized_data = []
for each in dataset.data:
  tokenized_data.append(process_and_tokenize(each, word_base=Wordbase.TEXT))

print(len(tokenized_data))

11314


# word2vec

In [0]:
import gensim.models.word2vec as w2v

model = w2v.Word2Vec(tokenized_data, 
                     min_count=3, 
                     window=5, 
                     size=100)

In [13]:
len(model.wv.vocab.keys())

46466

### Similarity 

In [14]:
model.wv.similar_by_word('Christ')

/usr/local/lib/python3.6/dist-packages/gensim/matutils.py:737: FutureWarning: Conversion of the second argument of issubdtype from `int` to `np.signedinteger` is deprecated. In future, it will be treated as `np.int64 == np.dtype(int).type`.
  if np.issubdtype(vec.dtype, np.int):


[('Lord', 0.9860788583755493),
 ('Father', 0.9811240434646606),
 ('Jesus', 0.9763765335083008),
 ('Spirit', 0.9741157293319702),
 ('Holy', 0.9707719087600708),
 ('resurrection', 0.9677433967590332),
 ('sin', 0.9661381244659424),
 ('Son', 0.9655351638793945),
 ('heaven', 0.9623531699180603),
 ('Matt51419', 0.9603991508483887)]

**Observation:** Note the `'Matt51419'` in the list – this seems some sort of mistake

# Mistake

The mistake is that we are feeding each of the articles as 1 single word corpus to wrod2vec – which means that if there are more than 1 sentences in the article – they are all being treated as 1 single text corpus of words which can occur together.

The above is fine for some cases – but the ideal input to a model like word2vec is the list of **sentences**, each of which is a list of tokens – as in most cases we are trying to learn the relationship between how words inside a setence (and not across sentences – words across 2 sentences don't necessarily have to have any semantic relation, as 2 sentences in 2 different paragraphs could be talking about something completely **unrelated**)


---

### Let's try again – this time with an input as a list of sentences, with each sentence being a list of tokens

In [16]:
from nltk import sent_tokenize

tokenized_data = []
for doc in dataset.data:
  for sent in sent_tokenize(doc):    
    tokenized_data.append(process_and_tokenize(sent, word_base=Wordbase.TEXT))

print(len(tokenized_data))

166032


In [0]:
import gensim.models.word2vec as w2v

model = w2v.Word2Vec(tokenized_data, 
                     min_count=3, 
                     window=5, 
                     size=100)

In [18]:
len(model.wv.vocab.keys())

46469

In [21]:
model.wv.similar_by_word('Christ')

/usr/local/lib/python3.6/dist-packages/gensim/matutils.py:737: FutureWarning: Conversion of the second argument of issubdtype from `int` to `np.signedinteger` is deprecated. In future, it will be treated as `np.int64 == np.dtype(int).type`.
  if np.issubdtype(vec.dtype, np.int):


[('Lord', 0.9780754446983337),
 ('Jesus', 0.9770290851593018),
 ('Father', 0.9720150828361511),
 ('Son', 0.9653409719467163),
 ('Spirit', 0.959479570388794),
 ('Bless', 0.9576663970947266),
 ('Lucifer', 0.9555511474609375),
 ('Holy', 0.9528343081474304),
 ('heaven', 0.946780264377594),
 ('bless', 0.9445289373397827)]

**Observation:** `Matt51419` is gone, and certainly this list of top10 similar words to `Christ` feels much more what it should be